# E12 — O esquecimento mínimo que ainda aprende

O capítulo anterior terminou numa frase que pede conta: a escala precisa ser atualizada sempre, e
atualizar é trocar passado por presente. **Quanto do passado se pode apagar antes que o sistema
deixe de aprender?**

Duas maneiras de esquecer entram na balança. A **mistura exponencial** dá peso à estimativa de
ontem e peso à observação de hoje, e a memória efetiva é o inverso do peso do presente. A **janela
deslizante** é a média dos últimos tantos dias: lembra exatamente esses dias e não lembra nada
antes deles. A janela é a referência declarada, e é a forma que o primeiro capítulo usou.

O mundo é o mesmo de sempre: uma oscilação que dobra de uma vez no meio da série. O que se mede é o
erro relativo da estimativa contra a verdade declarada, dia a dia, em vinte mundos sorteados.

A tolerância do capítulo entra em duas unidades, e elas ficam separadas no topo deste
caderno: o critério de viabilidade compara o erro com **quinze por cento do NÍVEL novo**, e a
proposição da meia-vida conta os dias até **a mesma tolerância em fração do DEGRAU**. Como o
mundo dobra de uma vez, o degrau é metade do nível, de modo que `TOLERANCIA_DEGRAU` vale o
dobro de `TOLERANCIA` — e foi por as duas serem o mesmo 0,15 que a prosa dizia uma unidade e
o critério media na outra.


In [1]:
# <- brinque com: TAXAS, JANELAS, HORIZONTES, TOLERANCIA, SEMENTES
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import esquecimento, graficos, mudanca

SIGMA = 0.01
VERDADE = SIGMA * np.sqrt(2.0 / np.pi)
DIAS = 14000
QUANDO = 4000
FATOR = 2.0
SEMENTES = 20
SEMENTE = 500
HORIZONTE = 250
HORIZONTES = (30, 45, 60, 90, 120, 250)
TOLERANCIA = 0.15  # fracao do NIVEL novo: e a unidade do erro relativo medido aqui
TOLERANCIA_DEGRAU = TOLERANCIA / (1.0 - 1.0 / FATOR)  # a MESMA tolerancia em fracao do degrau
MEMORIAS = (1000, 250, 50, 21, 10, 5, 2, 1)
TAXAS = tuple(sorted(round(1.0 / m, 4) for m in MEMORIAS))
JANELAS = MEMORIAS

series = [np.abs(mudanca.degrau(DIAS, np.random.default_rng(SEMENTE + i), fator=FATOR,
                                quando=QUANDO)) for i in range(SEMENTES)]
verdade = np.array([VERDADE * (FATOR if t >= QUANDO else 1.0) for t in range(DIAS)])
print("verdade antes %.5f | depois %.5f | series %d" % (VERDADE, VERDADE * FATOR, SEMENTES))
print("memorias: %s" % (MEMORIAS,))


verdade antes 0.00798 | depois 0.01596 | series 20
memorias: (1000, 250, 50, 21, 10, 5, 2, 1)


In [2]:
# As duas formas de esquecer, contra o mesmo degrau.
linhas = []
for memoria in MEMORIAS:
    taxa = round(1.0 / memoria, 4)
    e = [esquecimento.exponencial(s, taxa) for s in series]
    degrau = esquecimento.erro_varios(e, verdade, QUANDO, HORIZONTE)
    piso = esquecimento.erro_varios(e, verdade, QUANDO - 500, HORIZONTE)
    j = [esquecimento.janela(s, memoria) for s in series]
    jan = esquecimento.erro_varios(j, verdade, QUANDO, HORIZONTE)
    # O piso de ruido da propria janela: o capitulo 11.1 diz "a mesma janela erra por X no mundo
    # que nao muda", e o X que a tabela carregava era o piso da MISTURA, nao o da janela.
    piso_j = esquecimento.erro_varios(j, verdade, QUANDO - 500, HORIZONTE)
    linhas.append({"memoria": memoria, "taxa": taxa,
                   "exponencial": degrau[0], "exponencial_dp": degrau[1], "piso": piso[0],
                   "piso_janela": piso_j[0],
                   "janela": jan[0], "janela_dp": jan[1],
                   "dias_ate_15": esquecimento.dias_para_tolerancia(taxa, TOLERANCIA_DEGRAU)})
tabela = pd.DataFrame(linhas).set_index("memoria")
print(tabela.round(4).to_string())
print()
melhor = tabela["exponencial"].idxmin()
print("melhor exponencial: memoria %d (taxa %.4f) com erro %.4f" % (melhor, tabela.loc[melhor, "taxa"], tabela.loc[melhor, "exponencial"]))
print("melhor janela: memoria %d com erro %.4f" % (tabela["janela"].idxmin(), tabela["janela"].min()))
print("nunca esquecer (memoria mil): %.4f | esquecer tudo (memoria um): %.4f"
      % (tabela.loc[1000, "exponencial"], tabela.loc[1, "exponencial"]))


           taxa  exponencial  exponencial_dp    piso  piso_janela  janela  janela_dp  dias_ate_15
memoria                                                                                          
1000     0.0010       0.4405          0.0087  0.0164       0.0185  0.4327     0.0111    1203.3707
250      0.0040       0.3110          0.0177  0.0308       0.0399  0.2494     0.0257     300.3908
50       0.0200       0.1201          0.0209  0.0683       0.0967  0.1237     0.0177      59.5946
21       0.0476       0.1178          0.0176  0.1045       0.1433  0.1486     0.0233      24.6867
10       0.1000       0.1506          0.0175  0.1481       0.1996  0.1988     0.0175      11.4272
5        0.2000       0.2072          0.0143  0.2087       0.2752  0.2763     0.0188       5.3955
2        0.5000       0.3515          0.0142  0.3541       0.4356  0.4301     0.0210       1.7370
1        1.0000       0.6067          0.0299  0.6125       0.6121  0.6063     0.0301       0.0000

melhor exponencial:

In [3]:
# O que uma memoria consegue prometer: a tolerancia dela contra os dias que ela leva.
feixe = {}
for memoria in MEMORIAS:
    taxa = round(1.0 / memoria, 4)
    e = [esquecimento.exponencial(s, taxa) for s in series]
    for h in HORIZONTES:
        feixe[(memoria, h)] = esquecimento.erro_varios(e, verdade, QUANDO, h)[0]
quadro = pd.Series(feixe).unstack()
print("erro da exponencial, por memória e por horizonte de avaliacao:")
print(quadro.round(4).to_string())
print()
viaveis = [(m, h) for (m, h), v in feixe.items() if v <= TOLERANCIA]
print("pares (memoria, horizonte) que ficam dentro de %.0f%%: %d de %d"
      % (100 * TOLERANCIA, len(viaveis), len(feixe)))
if viaveis:
    mais_curto = min(viaveis, key=lambda p: p[1])
    mais_longa = max(viaveis, key=lambda p: p[0])
    print("horizonte mais curto que aprende: %d dias, com memoria %d (erro %.4f)"
          % (mais_curto[1], mais_curto[0], feixe[mais_curto]))
    print("memoria mais longa que aprende: %d dias, com horizonte %d (erro %.4f)"
          % (mais_longa[0], mais_longa[1], feixe[mais_longa]))
piso_melhor = tabela["exponencial"].min()
print("melhor erro de toda a varredura: %.4f" % piso_melhor)


erro da exponencial, por memória e por horizonte de avaliacao:
         30      45      60      90      120     250
1     0.5922  0.5978  0.6043  0.6065  0.6076  0.6067
2     0.3394  0.3456  0.3422  0.3484  0.3500  0.3515
5     0.2044  0.2088  0.2038  0.2073  0.2041  0.2072
10    0.1938  0.1784  0.1674  0.1631  0.1549  0.1506
21    0.2570  0.2104  0.1836  0.1570  0.1401  0.1178
50    0.3677  0.3180  0.2775  0.2226  0.1843  0.1201
250   0.4625  0.4483  0.4346  0.4096  0.3870  0.3110
1000  0.4903  0.4865  0.4826  0.4753  0.4682  0.4405

pares (memoria, horizonte) que ficam dentro de 15%: 3 de 48
horizonte mais curto que aprende: 120 dias, com memoria 21 (erro 0.1401)
memoria mais longa que aprende: 50 dias, com horizonte 250 (erro 0.1201)
melhor erro de toda a varredura: 0.1178


In [4]:
# Figura 1: o erro contra a memoria, nas duas formas de esquecer.
fig, eixo = plt.subplots(figsize=(8.6, 4.4))
raios = 1.959963985 / np.sqrt(SEMENTES)
eixo.errorbar(tabela.index, tabela["exponencial"], yerr=raios * tabela["exponencial_dp"], marker="o",
              capsize=3, color="#1f4e79", lw=1.6, label="mistura exponencial")
eixo.errorbar(tabela.index, tabela["janela"], yerr=raios * tabela["janela_dp"], marker="s",
              capsize=3, color="#b03a2e", lw=1.6, label="janela deslizante")
eixo.axhline(piso_melhor, color="#555555", ls=":", lw=1.2, label="o melhor de toda a varredura")
eixo.set_xscale("log")
eixo.set_xticks(list(MEMORIAS))
eixo.set_xticklabels([str(m) for m in MEMORIAS], fontsize=9)
eixo.set_xlabel("memória efetiva (dias)")
eixo.set_ylabel("erro relativo médio depois do degrau")
eixo.legend(frameon=False, fontsize=9)
eixo.grid(alpha=0.25)
fig.tight_layout()
graficos.salvar(fig, "E12_esquecimento", 1)
plt.close(fig)
print("erro no extremo esquecido: %.4f | no extremo que nunca esquece: %.4f"
      % (tabela.loc[1, "exponencial"], tabela.loc[1000, "exponencial"]))


erro no extremo esquecido: 0.6067 | no extremo que nunca esquece: 0.4405


In [5]:
# Figura 2: o que cada memoria promete, em tolerancia e em dias.
fig, eixo = plt.subplots(figsize=(8.6, 4.4))
eixo.plot(tabela["dias_ate_15"], tabela["exponencial"], marker="o", color="#1f4e79", lw=1.6,
          label="mistura exponencial")
eixo.plot(tabela["dias_ate_15"], tabela["piso"], marker="^", color="#2e7d32", lw=1.4, ls="--",
          label="o piso de ruído da mesma memória")
eixo.axhline(TOLERANCIA, color="#b03a2e", ls=":", lw=1.2, label="a tolerância declarada")
eixo.scatter([tabela.loc[melhor, "dias_ate_15"]], [tabela["exponencial"].min()], s=140, marker="*",
             color="#b03a2e", zorder=5, label="o melhor desenho")
eixo.set_xlabel("dias que a conta prevê até %.0f%% do nível novo" % (100 * TOLERANCIA))
eixo.set_ylabel("erro depois do degrau")
eixo.legend(frameon=False, fontsize=8)
eixo.grid(alpha=0.25)
fig.tight_layout()
graficos.salvar(fig, "E12_esquecimento", 2)
plt.close(fig)
print("dias previstos para a melhor memoria: %.1f" % tabela.loc[melhor, "dias_ate_15"])


dias previstos para a melhor memoria: 24.7


## Leitura visual das figuras

Feita nesta sessão abrindo os .png com a ponte de visão (AGENTS.md §9). Observação, não número.

**Figura 1.** Duas bacias em U e uma linha de referência. A azul tem o fundo mais à esquerda e
encosta na linha pontilhada do melhor de toda a varredura; a vermelha desce mais devagar e tem o
fundo mais à direita. Do lado da memória longa as duas sobem juntas, e do lado da memória curta
elas voltam a se encontrar, coladas — com um dia de memória as duas formas de esquecer são a mesma
coisa. O que o eixo engana: o horizontal é logarítmico, de modo que os dois últimos pontos parecem
vizinhos embora valham centenas de dias de distância.

**Figura 2.** Duas curvas no plano dos dias contra o erro, e elas se cruzam. A de cima tem fundo em
U; a tracejada do piso de ruído desce sempre. À esquerda as duas andam coladas, e ali o erro é todo
ruído; à direita elas se separam, e a distância entre elas é o atraso em relação à mudança. A linha
da tolerância declarada atravessa a figura, e a estrela do melhor fica logo abaixo dela. O que o eixo
engana: a horizontal é o que a conta **prevê**, e não o que se mede, de modo que a posição de cada
ponto é uma promessa e a altura é a entrega.


In [6]:
# O resultado: um objeto por grandeza, para o livro citar por comando.
NOMES = {1: "um", 2: "dois", 5: "cinco", 10: "dez", 21: "vinte_e_um", 50: "cinquenta",
         250: "duzentos_e_cinquenta", 1000: "mil"}
resultado = {
    "esquecimento_sementes": int(SEMENTES),
    "esquecimento_dias_de_serie": int(DIAS),
    "esquecimento_quando": int(QUANDO),
    "esquecimento_fator": float(FATOR),
    "esquecimento_horizonte": int(HORIZONTE),
    "esquecimento_tolerancia": float(TOLERANCIA),
    "esquecimento_tolerancia_degrau": float(TOLERANCIA_DEGRAU),
    "esquecimento_memorias": int(len(MEMORIAS)),
    "esquecimento_melhor_memoria": int(melhor),
    "esquecimento_melhor_taxa": float(tabela.loc[melhor, "taxa"]),
    "esquecimento_melhor": float(tabela.loc[melhor, "exponencial"]),
    "esquecimento_melhor_dispersao": float(tabela.loc[melhor, "exponencial_dp"]),
    "esquecimento_melhor_dias": float(tabela.loc[melhor, "dias_ate_15"]),
    "esquecimento_melhor_janela": int(tabela["janela"].idxmin()),
    "esquecimento_melhor_janela_erro": float(tabela["janela"].min()),
    "esquecimento_piso_na_melhor": float(tabela.loc[melhor, "piso"]),
    "esquecimento_piso_janela_duzentos_e_cinquenta": float(tabela.loc[250, "piso_janela"]),
    "esquecimento_pares_viaveis": int(len(viaveis)),
    "esquecimento_pares_todos": int(len(feixe)),
}
if viaveis:
    resultado["esquecimento_horizonte_curto"] = int(mais_curto[1])
    resultado["esquecimento_memoria_do_curto"] = int(mais_curto[0])
    resultado["esquecimento_erro_do_curto"] = float(feixe[mais_curto])
    resultado["esquecimento_memoria_longa"] = int(mais_longa[0])
    resultado["esquecimento_horizonte_da_longa"] = int(mais_longa[1])
for memoria in MEMORIAS:
    nome = NOMES[memoria]
    resultado["esquecimento_erro_%s" % nome] = float(tabela.loc[memoria, "exponencial"])
    resultado["esquecimento_janela_%s" % nome] = float(tabela.loc[memoria, "janela"])
    resultado["esquecimento_piso_%s" % nome] = float(tabela.loc[memoria, "piso"])
    resultado["esquecimento_dias_%s" % nome] = float(tabela.loc[memoria, "dias_ate_15"])
HORIZONTES_POR_EXTENSO = {30: "trinta", 45: "quarenta_e_cinco", 60: "sessenta", 90: "noventa",
                          120: "cento_e_vinte", 250: "duzentos_e_cinquenta"}
for h in HORIZONTES:
    resultado["esquecimento_melhor_no_horizonte_%s" % HORIZONTES_POR_EXTENSO[h]] = float(
        min(feixe[(m, h)] for m in MEMORIAS))
    melhor_do_horizonte = min(MEMORIAS, key=lambda m: feixe[(m, h)])
    resultado["esquecimento_memoria_do_horizonte_%s" % HORIZONTES_POR_EXTENSO[h]] = int(melhor_do_horizonte)

caminho = Path("lab/resultados/E12_esquecimento.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True),
                   encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))


lab/resultados/E12_esquecimento.json gravado | 68 grandezas
